In [1]:
print('hi')

hi


In [2]:
print(1+2)

3


In [3]:
print(1+2)

3


In [6]:
"""
Count the events in the files of each HNL sample, compute V2, tau and ctau from the formula
    tau = 4.49e-12 s * (M / 1 GeV)^-5.19 / |V|^2     (times 0.5 for Majorana)
and write a table with the masses along the top row and ctau, V2, tau, number of events down the rows.
Optionally upload the table to a Google Sheet.
 
Needs: pip install uproot pandas   (and gspread for the upload)
The event counting and the Google upload are not tested here: try them on one sample first.
"""
import os
import re
import glob
import json
import pandas as pd
import uproot
from concurrent.futures import ThreadPoolExecutor
 
SAMPLE_GLOB = "M-HeavyNeutrino_lljj_M*_V*_mu_NLO_Run3Summer24-150X_MiniAODv6"
TREE = "Events"
V_IS_V2 = False      # False: the number after "_V" in the folder name is |V|, True: it is already |V|^2
MAJORANA = True      # True: multiply the Dirac lifetime by 0.5 (check the convention of your model)
MASS_ACROSS = False   # True: masses along the top row, quantities down the first column; False: one row per sample
OUT_CSV = "hnl_samples.csv"
UPLOAD = False       # True: also write the table into a Google Sheet (see the block at the end)
 
C_CM_PER_S = 2.99792458e10
NAME_PATTERN = re.compile(r"_M(?P<m>[0-9.]+)_V(?P<v>[0-9.eE+-]+?)_mu_")
 
 
def hnl_tau_s(mass, v2, majorana=True):
    tau = 4.49e-12 * mass ** -5.19 / v2          # seconds, Dirac
    return 0.5 * tau if majorana else tau
 
 
def count_events(path):
    with uproot.open(path) as f:
        return f[TREE].num_entries
 
 

def safe_count(f):
    try:
        return f, count_events(f), None
    except Exception as e:                   # unreadable or corrupted file
        return f, 0, e

# ---------------- loop over the samples ----------------
rows = []
for d in sorted(glob.glob(os.path.join(BASE_DIR, SAMPLE_GLOB))):
    
    m = NAME_PATTERN.search(os.path.basename(d))
    if not m:
        print("cannot read mass/V from", d)
        continue
    mass = float(m["m"])
    
    files = sorted(glob.glob(os.path.join(d, "*.root")))
    with ThreadPoolExecutor(max_workers=8) as ex:
        results = list(ex.map(safe_count, files))

    n_events = sum(n for _, n, e in results if e is None)
    n_files = sum(e is None for _, _, e in results)
    #bad = [(f, e) for f, _, e in results if e is not None]
    print(f"{n_files} files, {n_events} events")
    v = float(m["v"])
    v2 = v if V_IS_V2 else v ** 2 
    tau = hnl_tau_s(mass, v2, MAJORANA)
    rows.append({"mass_GeV": mass, "V2": v2, "tau_s": tau, "ctau_cm": C_CM_PER_S * tau,
                 "n_events": n_events})
    break
 
df = pd.DataFrame(rows).sort_values(["mass_GeV", "V2"]).reset_index(drop=True)
 
# ---------------- layout ----------------
if MASS_ACROSS:
    dup = df["mass_GeV"].duplicated(keep=False).any()       # several V2 for the same mass?
    labels = [f"M={r.mass_GeV:g}, V2={r.V2:.3g}" if dup else f"M={r.mass_GeV:g}" for r in df.itertuples()]
    quantities = ["ctau_cm", "V2", "tau_s", "n_events"]
    table = {"quantity": quantities}
    for lab, r in zip(labels, df.itertuples()):
        table[lab] = [getattr(r, q) for q in quantities]
    out = pd.DataFrame(table, dtype=object)                 # object keeps the event counts as integers
else:
    rows_out = []
    for mass, g in df.groupby("mass_GeV"):
        rows_out.append({"mass_GeV": f"{mass:g}", "V2": "", "ctau_cm": "", "tau_s": "",
                         "n_events": int(g["n_events"].sum())})
        for r in g.itertuples():
            rows_out.append({"mass_GeV": "", "V2": f"{r.V2:.3e}", "ctau_cm": f"{r.ctau_cm:.4g}",
                             "tau_s": f"{r.tau_s:.3e}", "n_events": r.n_events})
    out = pd.DataFrame(rows_out, dtype=object)

out.to_csv(OUT_CSV, index=False)
print(out.to_string(index=False))
 
out.to_csv(OUT_CSV, index=False)
print(out.to_string(index=False))
print("written:", OUT_CSV)
 
# ---------------- optional: upload to a Google Sheet ----------------
# Simplest alternative: in Google Sheets use File > Import > Upload and pick the CSV.
if UPLOAD:
    import gspread
    gc = gspread.service_account(filename="service_account.json")   # key of a Google service account
    ws = gc.open("HNL samples").sheet1          # the sheet must be shared with the service-account e-mail
    ws.clear()
    values = [out.columns.tolist()] + json.loads(out.to_json(orient="values"))
    ws.update(range_name="A1", values=values)
    print("uploaded to the Google Sheet")
 


501 files, 1065124 events
mass_GeV        V2 ctau_cm     tau_s n_events
      10                              1065124
         2.161e-08   20.11 6.708e-10  1065124
mass_GeV        V2 ctau_cm     tau_s n_events
      10                              1065124
         2.161e-08   20.11 6.708e-10  1065124
written: hnl_samples.csv
